In [1]:
import pandas as pd
import tensorflow as tf
import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
#Evaluation cell
# it is a testData set
np.random.seed(69)
tf.random.set_seed(69)
hidden_data = pd.read_csv("testSet.csv")
input = ["nino_tminus2", "nino_tminus1", "nino_t"]
target = ["nino_tplus1", "nino_tplus2", "nino_tplus3", 
          "nino_tplus4", "nino_tplus5", "nino_tplus6"]
X_test = hidden_data[input]
scaler = joblib.load('input_scaler.pkl')
X_trans = scaler.transform(X_test)
actual = hidden_data[target].values

weights = tf.constant([0.3, 0.25, 0.2, 0.15, 0.07, 0.03], dtype=tf.float32)

def weighted_mse(y_true, y_pred):
    squared_error = tf.square(y_true - y_pred)
    weighted_error = squared_error * weights
    return tf.reduce_mean(tf.reduce_sum(weighted_error, axis=1))

# 1 Month
Ev_model1 = tf.keras.models.load_model('nino_model1.keras')
Ev_model2 = tf.keras.models.load_model('nino_model2.keras')
Ev_model3 = tf.keras.models.load_model('nino_model3.keras')
Ev_model4 = tf.keras.models.load_model('nino_model4.keras')
Ev_model5 = tf.keras.models.load_model('nino_model5.keras')
Ev_model6 = tf.keras.models.load_model('nino_model6.keras')
Ev_model_transfer = tf.keras.models.load_model('Transfered_model_tplus6.keras')

# 6 Months
Ev_model_B = tf.keras.models.load_model('MOmodel.keras')
Ev_model_weighted_B = tf.keras.models.load_model('Weighted_MOmodel.keras', custom_objects={'weighted_mse': weighted_mse})

pred1 = Ev_model1(X_trans)

pred2 = Ev_model2(X_trans)

pred3 = Ev_model3(X_trans)

pred4 = Ev_model4(X_trans)

pred5 = Ev_model5(X_trans)

pred6 = Ev_model6(X_trans)

pred7 = Ev_model_transfer(X_trans)

pred8 = Ev_model_B(X_trans)

pred9 = Ev_model_weighted_B(X_trans)
forecast_horizons = ["t+1","t+2","t+3","t+4","t+5","t+6"]
single_preds = [pred1, pred2, pred3, pred4, pred5, pred6]
single_names = ["t+1","t+2","t+3","t+4","t+5","t+6","Transfer t+6"]

print("SINGLE-OUTPUT MODELS EVALUATION")
print("-"*40)
for i, pred in enumerate(single_preds):
    actual_i = actual[:,i]  # column i
    pred_i = pred.numpy().flatten()
    rmse = np.sqrt(mean_squared_error(actual_i, pred_i))
    corr = np.corrcoef(pred_i, actual_i)[0,1]
    print(f"{single_names[i]}: RMSE={rmse:.4f}, Pearson={corr:.4f}")
    
print("\n" + "="*50 + "\n")

print("Transfer MODELS EVALUATION")
print("-"*40)
actual_i = actual[:, 5]  # last column t+6
pred_i = pred7.numpy().flatten()
rmse = np.sqrt(mean_squared_error(actual_i, pred_i))
corr = np.corrcoef(pred_i, actual_i)[0,1]
print(f"Transfer t+6: RMSE={rmse:.4f}, Pearson={corr:.4f}")

# 2. Multi-output models
multi_preds = [pred8, pred9]
multi_names = ["MOmodel", "Weighted MOmodel"]

print("MULTI-OUTPUT MODELS EVALUATION")
print("-"*40)

for j, pred in enumerate(multi_preds):
    print(f"{multi_names[j]}:")
    pred_np = pred.numpy()
    for i in range(pred_np.shape[1]):
        actual_i = actual[:,i]
        pred_i = pred_np[:,i]
        rmse = np.sqrt(mean_squared_error(actual_i, pred_i))
        corr = np.corrcoef(pred_i, actual_i)[0,1]
        print(f"  {forecast_horizons[i]}: RMSE={rmse:.4f}, Pearson={corr:.4f}")
    print("-"*40)

SINGLE-OUTPUT MODELS EVALUATION
----------------------------------------
t+1: RMSE=0.2008, Pearson=0.9803
t+2: RMSE=0.3620, Pearson=0.9232
t+3: RMSE=0.5414, Pearson=0.8209
t+4: RMSE=0.7490, Pearson=0.6067
t+5: RMSE=0.7945, Pearson=0.6120
t+6: RMSE=0.8672, Pearson=0.5246


Transfer MODELS EVALUATION
----------------------------------------
Transfer t+6: RMSE=0.9091, Pearson=0.6086
MULTI-OUTPUT MODELS EVALUATION
----------------------------------------
MOmodel:
  t+1: RMSE=0.2083, Pearson=0.9808
  t+2: RMSE=0.3949, Pearson=0.9229
  t+3: RMSE=0.5675, Pearson=0.8265
  t+4: RMSE=0.7013, Pearson=0.7090
  t+5: RMSE=0.7888, Pearson=0.6121
  t+6: RMSE=0.8649, Pearson=0.5034
----------------------------------------
Weighted MOmodel:
  t+1: RMSE=0.2047, Pearson=0.9788
  t+2: RMSE=0.3919, Pearson=0.9208
  t+3: RMSE=0.5544, Pearson=0.8284
  t+4: RMSE=0.6964, Pearson=0.7081
  t+5: RMSE=0.7912, Pearson=0.6052
  t+6: RMSE=0.8517, Pearson=0.5177
----------------------------------------
